In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════
# LOAD BOTH TRADE SETS — v3.7.1
# ═══════════════════════════════════════════════════════

files = {
    'AXS':   r'C:\Users\User\Downloads\4Pv3.7.1-S_BYBIT_AXSUSDT.P_2026-02-07_740ab.csv',
    'RIVER': r'C:\Users\User\Downloads\4Pv3.7.1-S_BYBIT_RIVERUSDT.P_2026-02-07_902f1.csv',
}

LEVERAGE = 20
all_trades = {}

for sym, path in files.items():
    df = pd.read_csv(path, encoding='utf-8-sig')
    
    exits = df[df['Type'].str.contains('Exit', case=False)].copy()
    entries = df[df['Type'].str.contains('Entry', case=False)].copy()
    
    trades = exits[['Trade #', 'Type', 'Signal', 'Date and time', 'Price USDT',
                     'Net P&L USDT', 'Net P&L %',
                     'Favorable excursion USDT', 'Favorable excursion %',
                     'Adverse excursion USDT', 'Adverse excursion %',
                     'Cumulative P&L USDT', 'Cumulative P&L %']].copy()
    
    entries_info = entries[['Trade #', 'Date and time', 'Price USDT', 'Type']].copy()
    entries_info.columns = ['Trade #', 'Entry Time', 'Entry Price', 'Entry Type']
    trades = trades.merge(entries_info, on='Trade #', how='left')
    trades.rename(columns={'Date and time': 'Exit Time', 'Price USDT': 'Exit Price', 'Type': 'Exit Type'}, inplace=True)
    
    trades['Direction'] = trades['Entry Type'].apply(lambda x: 'Long' if 'long' in x.lower() else 'Short')
    trades['PnL_20x'] = trades['Net P&L USDT'] * LEVERAGE
    trades['PnL_pct_20x'] = trades['Net P&L %'] * LEVERAGE
    trades['CumPnL_20x'] = trades['PnL_20x'].cumsum()
    trades['MFE_20x'] = trades['Favorable excursion USDT'] * LEVERAGE
    trades['MAE_20x'] = trades['Adverse excursion USDT'] * LEVERAGE
    trades['Win'] = trades['PnL_20x'] > 0
    trades['Loss'] = trades['PnL_20x'] < 0
    trades['Symbol'] = sym
    
    trades['Entry_dt'] = pd.to_datetime(trades['Entry Time'])
    trades['Exit_dt'] = pd.to_datetime(trades['Exit Time'])
    trades['Duration'] = trades['Exit_dt'] - trades['Entry_dt']
    trades['Duration_mins'] = trades['Duration'].dt.total_seconds() / 60
    trades['Entry_hour'] = trades['Entry_dt'].dt.hour
    trades['DOW'] = trades['Entry_dt'].dt.day_name()
    
    all_trades[sym] = trades.reset_index(drop=True)
    print(f'[{sym}] Loaded {len(trades)} trades  |  {trades["Entry Time"].iloc[0]} -> {trades["Exit Time"].iloc[-1]}')

symbols = list(all_trades.keys())
sym_colors = {'AXS': '#5dade2', 'RIVER': '#f39c12'}
print(f'\nTotal: {sum(len(t) for t in all_trades.values())} trades across {len(all_trades)} symbols')

In [ ]:
# ═══════════════════════════════════════════════════════
# CONSECUTIVE LOSS / WIN STREAK ANALYSIS
# ═══════════════════════════════════════════════════════

def compute_streaks(series):
    streaks = []
    current = 0
    for val in series:
        if val:
            current += 1
        else:
            if current > 0:
                streaks.append(current)
            current = 0
    if current > 0:
        streaks.append(current)
    return streaks

for sym in symbols:
    trades = all_trades[sym]
    loss_streaks = compute_streaks(trades['Loss'])
    win_streaks = compute_streaks(trades['Win'])
    
    streak_vals = []
    streak = 0
    for i in range(len(trades)):
        if trades.iloc[i]['Loss']:
            streak = streak - 1 if streak < 0 else -1
        elif trades.iloc[i]['Win']:
            streak = streak + 1 if streak > 0 else 1
        else:
            streak = 0
        streak_vals.append(streak)
    trades['streak'] = streak_vals
    all_trades[sym] = trades
    
    max_consec_losses = max(loss_streaks) if loss_streaks else 0
    max_consec_wins = max(win_streaks) if win_streaks else 0
    avg_loss_streak = np.mean(loss_streaks) if loss_streaks else 0
    avg_win_streak = np.mean(win_streaks) if win_streaks else 0
    
    worst_streak_end = trades['streak'].idxmin()
    worst_streak_len = abs(trades.loc[worst_streak_end, 'streak'])
    worst_streak_start = worst_streak_end - int(worst_streak_len) + 1
    worst_streak_trades = trades.iloc[worst_streak_start:worst_streak_end+1]
    worst_streak_damage = worst_streak_trades['PnL_20x'].sum()
    
    print(f'============================================================')
    print(f'  STREAK ANALYSIS — {sym}USDT.P')
    print(f'============================================================')
    print(f'  Max Consecutive Losses:  {max_consec_losses:>4}')
    print(f'  Max Consecutive Wins:    {max_consec_wins:>4}')
    print(f'  Avg Loss Streak:         {avg_loss_streak:>6.1f}')
    print(f'  Avg Win Streak:          {avg_win_streak:>6.1f}')
    print(f'  Total Loss Streaks:      {len(loss_streaks):>4}')
    print(f'  Total Win Streaks:       {len(win_streaks):>4}')
    print(f'  ---')
    print(f'  WORST STREAK: {int(worst_streak_len)} consecutive losses')
    print(f'  Damage: ${worst_streak_damage:>10.2f} (20x)')
    print(f'  Trades #{trades.iloc[worst_streak_start]["Trade #"]:.0f} - #{trades.iloc[worst_streak_end]["Trade #"]:.0f}')
    print(f'  From: {worst_streak_trades["Entry Time"].iloc[0]}')
    print(f'  To:   {worst_streak_trades["Exit Time"].iloc[-1]}')
    
    loss_dist = Counter(loss_streaks)
    print(f'\n  Loss Streak Distribution:')
    for length in sorted(loss_dist.keys()):
        bar = '#' * loss_dist[length]
        print(f'    {length:>2} in a row: {loss_dist[length]:>3}x  {bar}')
    print()

In [ ]:
# ═══════════════════════════════════════════════════════
# CORE PERFORMANCE METRICS (20x) — SIDE BY SIDE
# ═══════════════════════════════════════════════════════

# Commission: 0.04%/side on $10k notional = $4/side = $8 round trip
COMM_RT = 8

summary_rows = []

for sym in symbols:
    trades = all_trades[sym]
    total_trades = len(trades)
    winners = trades[trades['Win']]
    losers = trades[trades['Loss']]
    breakeven = trades[(~trades['Win']) & (~trades['Loss'])]
    
    win_rate = len(winners) / total_trades * 100
    total_pnl = trades['PnL_20x'].sum()
    avg_win = winners['PnL_20x'].mean() if len(winners) > 0 else 0
    avg_loss = losers['PnL_20x'].mean() if len(losers) > 0 else 0
    largest_win = trades['PnL_20x'].max()
    largest_loss = trades['PnL_20x'].min()
    median_win = winners['PnL_20x'].median() if len(winners) > 0 else 0
    median_loss = losers['PnL_20x'].median() if len(losers) > 0 else 0
    
    gross_profit = winners['PnL_20x'].sum() if len(winners) > 0 else 0
    gross_loss = abs(losers['PnL_20x'].sum()) if len(losers) > 0 else 1
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    payoff_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else float('inf')
    expectancy = total_pnl / total_trades
    
    longs = trades[trades['Direction'] == 'Long']
    shorts = trades[trades['Direction'] == 'Short']
    long_wr = longs['Win'].sum() / len(longs) * 100 if len(longs) > 0 else 0
    short_wr = shorts['Win'].sum() / len(shorts) * 100 if len(shorts) > 0 else 0
    long_pnl = longs['PnL_20x'].sum()
    short_pnl = shorts['PnL_20x'].sum()
    
    days = (trades['Exit_dt'].iloc[-1] - trades['Entry_dt'].iloc[0]).days
    trades_per_day = total_trades / max(days, 1)
    
    commission_total = total_trades * COMM_RT
    net_after_comm = total_pnl - commission_total
    
    summary_rows.append({
        'Symbol': sym,
        'Trades': total_trades,
        'Days': days,
        'Trades/Day': round(trades_per_day, 1),
        'Win Rate %': round(win_rate, 1),
        'Net P&L $': round(total_pnl, 2),
        'Commission $': round(commission_total, 2),
        'Net After Comm $': round(net_after_comm, 2),
        'Profit Factor': round(profit_factor, 3),
        'Payoff Ratio': round(payoff_ratio, 3),
        'Expectancy $/trade': round(expectancy, 2),
        'Avg Win $': round(avg_win, 2),
        'Avg Loss $': round(avg_loss, 2),
        'Median Win $': round(median_win, 2),
        'Median Loss $': round(median_loss, 2),
        'Largest Win $': round(largest_win, 2),
        'Largest Loss $': round(largest_loss, 2),
        'Long WR %': round(long_wr, 1),
        'Short WR %': round(short_wr, 1),
        'Long P&L $': round(long_pnl, 2),
        'Short P&L $': round(short_pnl, 2)
    })
    
    print(f'============================================================')
    print(f'  FOUR PILLARS v3.7.1 — {sym}USDT.P  |  20x Leverage')
    print(f'============================================================')
    print(f'  Total Trades:        {total_trades:>6}  ({trades_per_day:.1f}/day over {days}d)')
    print(f'  Winners:             {len(winners):>6}  ({win_rate:.1f}%)')
    print(f'  Losers:              {len(losers):>6}  ({100-win_rate:.1f}%)')
    print(f'  Breakeven:           {len(breakeven):>6}')
    print(f'  ---')
    print(f'  Net P&L (20x):     ${total_pnl:>10.2f}')
    print(f'  Commission Est:    -${commission_total:>10.2f}  (${COMM_RT} x {total_trades})')
    print(f'  NET AFTER COMM:    ${net_after_comm:>10.2f}')
    print(f'  Gross Profit:      ${gross_profit:>10.2f}')
    print(f'  Gross Loss:       -${gross_loss:>10.2f}')
    print(f'  Profit Factor:      {profit_factor:>10.3f}')
    print(f'  Payoff Ratio:       {payoff_ratio:>10.3f}  (avg W / avg L)')
    print(f'  Expectancy/Trade:  ${expectancy:>10.2f}')
    print(f'  ---')
    print(f'  Avg Winner:        ${avg_win:>10.2f}   Median: ${median_win:>8.2f}')
    print(f'  Avg Loser:         ${avg_loss:>10.2f}   Median: ${median_loss:>8.2f}')
    print(f'  Largest Win:       ${largest_win:>10.2f}')
    print(f'  Largest Loss:      ${largest_loss:>10.2f}')
    print(f'  ---')
    print(f'  LONG  trades: {len(longs):>5}  WR: {long_wr:.1f}%  P&L: ${long_pnl:>10.2f}')
    print(f'  SHORT trades: {len(shorts):>5}  WR: {short_wr:.1f}%  P&L: ${short_pnl:>10.2f}')
    print()

# Side-by-side comparison table
comp = pd.DataFrame(summary_rows).set_index('Symbol').T
print('\n=== SIDE-BY-SIDE COMPARISON ===')
print(comp.to_string())

In [ ]:
# ═══════════════════════════════════════════════════════
# DRAWDOWN & RISK METRICS
# ═══════════════════════════════════════════════════════

for sym in symbols:
    trades = all_trades[sym]
    equity = trades['CumPnL_20x']
    running_max = equity.cummax()
    drawdown = equity - running_max
    trades['Drawdown_20x'] = drawdown.values
    all_trades[sym] = trades
    
    max_dd = drawdown.min()
    max_dd_idx = drawdown.idxmin()
    max_dd_trade = trades.loc[max_dd_idx, 'Trade #']
    peak_before_dd = running_max.loc[:max_dd_idx].max()
    
    post_dd = equity.loc[max_dd_idx:]
    recovered = post_dd[post_dd >= peak_before_dd]
    if len(recovered) > 0:
        recovery_trade = trades.loc[recovered.index[0], 'Trade #']
        recovery_trades_needed = recovered.index[0] - max_dd_idx
        recovery_status = f'Recovered at trade #{recovery_trade:.0f} ({recovery_trades_needed} trades later)'
    else:
        recovery_status = 'NOT RECOVERED'
    
    total_pnl = trades['PnL_20x'].sum()
    sharpe_like = trades['PnL_20x'].mean() / trades['PnL_20x'].std() if trades['PnL_20x'].std() > 0 else 0
    downside = trades[trades['PnL_20x'] < 0]['PnL_20x']
    sortino_like = trades['PnL_20x'].mean() / downside.std() if len(downside) > 0 and downside.std() > 0 else 0
    recovery_factor = total_pnl / abs(max_dd) if max_dd != 0 else 0
    
    exit_types = trades.groupby('Signal').agg(
        count=('PnL_20x', 'count'),
        total_pnl=('PnL_20x', 'sum'),
        avg_pnl=('PnL_20x', 'mean'),
        win_rate=('Win', 'mean')
    ).round(2)
    exit_types['win_rate'] = (exit_types['win_rate'] * 100).round(1)
    
    print(f'============================================================')
    print(f'  RISK METRICS — {sym}USDT.P')
    print(f'============================================================')
    print(f'  Max Drawdown (20x):  ${max_dd:>10.2f}  (at trade #{max_dd_trade:.0f})')
    print(f'  Peak Before DD:      ${peak_before_dd:>10.2f}')
    print(f'  Recovery:  {recovery_status}')
    print(f'  Recovery Factor:     {recovery_factor:>10.3f}  (net P&L / max DD)')
    print(f'  ---')
    print(f'  Sharpe-like:         {sharpe_like:>10.4f}  (mean/std per trade)')
    print(f'  Sortino-like:        {sortino_like:>10.4f}  (mean/downside std)')
    
    print(f'\n  Exit Signal Breakdown ({sym}):')
    print(exit_types.to_string())
    print()

In [ ]:
# ═══════════════════════════════════════════════════════
# VISUAL DASHBOARD — DUAL SYMBOL
# ═══════════════════════════════════════════════════════

def style_ax(ax, title):
    ax.set_facecolor('#16213e')
    ax.set_title(title, color='white', fontsize=12, fontweight='bold', pad=10)
    ax.tick_params(colors='#aaa')
    for spine in ax.spines.values():
        spine.set_color('#333')
    ax.grid(True, alpha=0.15, color='white')

fig = plt.figure(figsize=(20, 24), facecolor='#1a1a2e')
gs = GridSpec(5, 2, figure=fig, hspace=0.4, wspace=0.3)

# 1. Equity Curves Overlaid
ax1 = fig.add_subplot(gs[0, :])
for sym in symbols:
    t = all_trades[sym]
    ax1.plot(t['CumPnL_20x'].values, color=sym_colors[sym], linewidth=1.2,
             label=f'{sym} (${t["PnL_20x"].sum():.0f})', alpha=0.9)
ax1.axhline(y=0, color='#666', linestyle='--', linewidth=0.8)
style_ax(ax1, 'EQUITY CURVES (20x Leverage)')
ax1.set_xlabel('Trade #', color='#aaa')
ax1.set_ylabel('Cumulative P&L ($)', color='#aaa')
ax1.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=11)

# 2. Drawdown Curves Overlaid
ax2 = fig.add_subplot(gs[1, :])
for sym in symbols:
    t = all_trades[sym]
    ax2.fill_between(range(len(t)), t['Drawdown_20x'], alpha=0.3, color=sym_colors[sym])
    ax2.plot(t['Drawdown_20x'].values, color=sym_colors[sym], linewidth=0.8,
             label=f'{sym} (max ${t["Drawdown_20x"].min():.0f})')
style_ax(ax2, 'DRAWDOWN CURVES (20x)')
ax2.set_xlabel('Trade #', color='#aaa')
ax2.set_ylabel('Drawdown ($)', color='#aaa')
ax2.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=11)

# 3-4. P&L per trade bars (one per symbol)
for i, sym in enumerate(symbols):
    ax = fig.add_subplot(gs[2, i])
    t = all_trades[sym]
    colors = ['#00d4aa' if x > 0 else '#ff4757' for x in t['PnL_20x']]
    ax.bar(range(len(t)), t['PnL_20x'], color=colors, width=1.0, alpha=0.7)
    style_ax(ax, f'P&L PER TRADE — {sym} (20x)')
    ax.set_xlabel('Trade #', color='#aaa')
    ax.set_ylabel('P&L ($)', color='#aaa')

# 5-6. P&L Histograms
for i, sym in enumerate(symbols):
    ax = fig.add_subplot(gs[3, i])
    t = all_trades[sym]
    ax.hist(t['PnL_20x'], bins=50, color=sym_colors[sym], alpha=0.7, edgecolor='#333')
    ax.axvline(x=0, color='white', linestyle='--', linewidth=0.8)
    mean_pnl = t['PnL_20x'].mean()
    med_pnl = t['PnL_20x'].median()
    ax.axvline(x=mean_pnl, color='#ffd32a', linestyle='-', linewidth=1.5, label=f'Mean: ${mean_pnl:.2f}')
    ax.axvline(x=med_pnl, color='#ff4757', linestyle='-', linewidth=1.5, label=f'Median: ${med_pnl:.2f}')
    style_ax(ax, f'P&L DISTRIBUTION — {sym}')
    ax.set_xlabel('P&L ($)', color='#aaa')
    ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=8)

# 7-8. Streak charts
for i, sym in enumerate(symbols):
    ax = fig.add_subplot(gs[4, i])
    t = all_trades[sym]
    streak_colors = ['#00d4aa' if s > 0 else '#ff4757' if s < 0 else '#666' for s in t['streak']]
    ax.bar(range(len(t)), t['streak'], color=streak_colors, width=1.0, alpha=0.7)
    style_ax(ax, f'WIN/LOSS STREAKS — {sym}')
    ax.set_xlabel('Trade #', color='#aaa')
    ax.set_ylabel('Streak Length', color='#aaa')
    ax.axhline(y=0, color='#666', linewidth=0.5)

fig.suptitle('FOUR PILLARS v3.7.1 — AXS vs RIVER  |  TRADE DASHBOARD',
             color='white', fontsize=16, fontweight='bold', y=0.99)

plt.savefig(r'C:\Users\User\Documents\Obsidian Vault\07-TEMPLATES\trade_dashboard_v371.png',
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('Dashboard saved to trade_dashboard_v371.png')

In [ ]:
# ═══════════════════════════════════════════════════════
# TIME & DURATION ANALYSIS
# ═══════════════════════════════════════════════════════

for sym in symbols:
    trades = all_trades[sym]
    winners = trades[trades['Win']]
    losers = trades[trades['Loss']]
    
    w_dur = winners['Duration_mins'].mean() if len(winners) > 0 else 0
    l_dur = losers['Duration_mins'].mean() if len(losers) > 0 else 0
    
    print(f'============================================================')
    print(f'  TIME ANALYSIS — {sym}USDT.P')
    print(f'============================================================')
    print(f'  Avg Trade Duration:  {trades["Duration_mins"].mean():>8.1f} min')
    print(f'  Median Duration:     {trades["Duration_mins"].median():>8.1f} min')
    print(f'  Shortest Trade:      {trades["Duration_mins"].min():>8.1f} min')
    print(f'  Longest Trade:       {trades["Duration_mins"].max():>8.1f} min')
    print(f'  Avg Winner Duration: {w_dur:>8.1f} min')
    print(f'  Avg Loser Duration:  {l_dur:>8.1f} min')
    
    hourly = trades.groupby('Entry_hour').agg(
        count=('PnL_20x', 'count'),
        total_pnl=('PnL_20x', 'sum'),
        avg_pnl=('PnL_20x', 'mean'),
        win_rate=('Win', 'mean')
    ).round(2)
    hourly['win_rate'] = (hourly['win_rate'] * 100).round(1)
    
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    daily = trades.groupby('DOW').agg(
        count=('PnL_20x', 'count'),
        total_pnl=('PnL_20x', 'sum'),
        avg_pnl=('PnL_20x', 'mean'),
        win_rate=('Win', 'mean')
    ).round(2)
    daily['win_rate'] = (daily['win_rate'] * 100).round(1)
    daily = daily.reindex([d for d in day_order if d in daily.index])
    
    print(f'\n  P&L by Hour (UTC) — {sym}:')
    print(hourly.to_string())
    print(f'\n  P&L by Day of Week — {sym}:')
    print(daily.to_string())
    print()

In [ ]:
# ═══════════════════════════════════════════════════════
# MFE / MAE ANALYSIS (20x) — BOTH SYMBOLS
# ═══════════════════════════════════════════════════════

fig2, axes = plt.subplots(2, 2, figsize=(18, 12), facecolor='#1a1a2e')

for row, sym in enumerate(symbols):
    trades = all_trades[sym]
    win_mask = trades['Win']
    
    # MFE vs Actual P&L
    ax = axes[row, 0]
    ax.set_facecolor('#16213e')
    ax.scatter(trades.loc[win_mask, 'MFE_20x'], trades.loc[win_mask, 'PnL_20x'],
               c='#00d4aa', alpha=0.4, s=15, label='Winners')
    ax.scatter(trades.loc[~win_mask, 'MFE_20x'], trades.loc[~win_mask, 'PnL_20x'],
               c='#ff4757', alpha=0.4, s=15, label='Losers')
    ax.axhline(y=0, color='#666', linestyle='--', linewidth=0.8)
    max_mfe = trades['MFE_20x'].max()
    ax.plot([0, max_mfe], [0, max_mfe], color='#ffd32a', linestyle=':', alpha=0.5, label='100% capture')
    ax.set_title(f'MFE vs P&L — {sym} (20x)', color='white', fontweight='bold')
    ax.set_xlabel('MFE ($)', color='#aaa')
    ax.set_ylabel('P&L ($)', color='#aaa')
    ax.tick_params(colors='#aaa')
    ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=8)
    ax.grid(True, alpha=0.15)
    for spine in ax.spines.values(): spine.set_color('#333')
    
    # MAE vs Actual P&L
    ax = axes[row, 1]
    ax.set_facecolor('#16213e')
    ax.scatter(trades.loc[win_mask, 'MAE_20x'], trades.loc[win_mask, 'PnL_20x'],
               c='#00d4aa', alpha=0.4, s=15, label='Winners')
    ax.scatter(trades.loc[~win_mask, 'MAE_20x'], trades.loc[~win_mask, 'PnL_20x'],
               c='#ff4757', alpha=0.4, s=15, label='Losers')
    ax.axhline(y=0, color='#666', linestyle='--', linewidth=0.8)
    ax.set_title(f'MAE vs P&L — {sym} (20x)', color='white', fontweight='bold')
    ax.set_xlabel('MAE ($)', color='#aaa')
    ax.set_ylabel('P&L ($)', color='#aaa')
    ax.tick_params(colors='#aaa')
    ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=8)
    ax.grid(True, alpha=0.15)
    for spine in ax.spines.values(): spine.set_color('#333')

fig2.suptitle('TRADE EFFICIENCY — MFE / MAE  |  AXS vs RIVER', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\User\Documents\Obsidian Vault\07-TEMPLATES\mfe_mae_v371.png',
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

for sym in symbols:
    trades = all_trades[sym]
    winners = trades[trades['Win']]
    losers = trades[trades['Loss']]
    
    if len(winners) > 0:
        capture_eff = (winners['PnL_20x'] / winners['MFE_20x'].replace(0, np.nan)).dropna().mean() * 100
        print(f'[{sym}] MFE Capture Efficiency (winners): {capture_eff:.1f}%')
    
    losers_had_profit = (losers['MFE_20x'] > 0).sum()
    pct = losers_had_profit / len(losers) * 100 if len(losers) > 0 else 0
    print(f'[{sym}] Losers that were profitable at some point: {losers_had_profit}/{len(losers)} ({pct:.1f}%)')
    if losers_had_profit > 0:
        avg_missed = losers[losers['MFE_20x'] > 0]['MFE_20x'].mean()
        print(f'  -> Avg missed profit on those losers: ${avg_missed:.2f}')
    print()

In [ ]:
# ═══════════════════════════════════════════════════════
# ROLLING PERFORMANCE (50-trade window)
# ═══════════════════════════════════════════════════════

window = 50

fig3, axes = plt.subplots(3, 2, figsize=(20, 14), facecolor='#1a1a2e')

for col, sym in enumerate(symbols):
    trades = all_trades[sym]
    trades['Rolling_WR'] = trades['Win'].rolling(window).mean() * 100
    trades['Rolling_PnL'] = trades['PnL_20x'].rolling(window).sum()
    trades['Rolling_Avg'] = trades['PnL_20x'].rolling(window).mean()
    
    # Rolling win rate
    ax = axes[0, col]
    ax.set_facecolor('#16213e')
    ax.plot(trades['Rolling_WR'], color=sym_colors[sym], linewidth=1.2)
    ax.axhline(y=50, color='#ff4757', linestyle='--', linewidth=0.8, label='50% WR')
    style_ax(ax, f'Rolling {window}-Trade WR — {sym}')
    ax.set_ylabel('Win Rate %', color='#aaa')
    ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white')
    
    # Rolling P&L sum
    ax = axes[1, col]
    ax.set_facecolor('#16213e')
    ax.fill_between(range(len(trades)), trades['Rolling_PnL'].fillna(0), alpha=0.3, color=sym_colors[sym])
    ax.plot(trades['Rolling_PnL'], color=sym_colors[sym], linewidth=1.0)
    ax.axhline(y=0, color='#ff4757', linestyle='--', linewidth=0.8)
    style_ax(ax, f'Rolling {window}-Trade P&L — {sym}')
    ax.set_ylabel('P&L ($)', color='#aaa')
    
    # Rolling avg
    ax = axes[2, col]
    ax.set_facecolor('#16213e')
    ax.plot(trades['Rolling_Avg'], color=sym_colors[sym], linewidth=1.0)
    ax.axhline(y=0, color='#ff4757', linestyle='--', linewidth=0.8)
    style_ax(ax, f'Rolling {window}-Trade Avg — {sym}')
    ax.set_xlabel('Trade #', color='#aaa')
    ax.set_ylabel('Avg P&L ($)', color='#aaa')

fig3.suptitle('ROLLING PERFORMANCE  |  AXS vs RIVER', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\User\Documents\Obsidian Vault\07-TEMPLATES\rolling_v371.png',
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# EXECUTIVE SUMMARY + TOP/WORST TRADES
# ═══════════════════════════════════════════════════════

for sym in symbols:
    trades = all_trades[sym]
    total_trades = len(trades)
    total_pnl = trades['PnL_20x'].sum()
    winners = trades[trades['Win']]
    losers = trades[trades['Loss']]
    win_rate = len(winners) / total_trades * 100
    gross_profit = winners['PnL_20x'].sum() if len(winners) > 0 else 0
    gross_loss = abs(losers['PnL_20x'].sum()) if len(losers) > 0 else 1
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    expectancy = total_pnl / total_trades
    max_dd = trades['Drawdown_20x'].min()
    commission_total = total_trades * COMM_RT
    net_after_comm = total_pnl - commission_total
    days = (trades['Exit_dt'].iloc[-1] - trades['Entry_dt'].iloc[0]).days
    
    loss_streaks = compute_streaks(trades['Loss'])
    max_consec_losses = max(loss_streaks) if loss_streaks else 0
    worst_streak_end = trades['streak'].idxmin()
    worst_streak_len = abs(trades.loc[worst_streak_end, 'streak'])
    worst_streak_start = worst_streak_end - int(worst_streak_len) + 1
    worst_streak_damage = trades.iloc[worst_streak_start:worst_streak_end+1]['PnL_20x'].sum()
    
    print(f'\n=== TOP 10 BEST TRADES — {sym} (20x) ===')
    best = trades.nlargest(10, 'PnL_20x')[['Trade #', 'Direction', 'Entry Time', 'Exit Time', 'Signal', 'PnL_20x', 'Duration_mins']].copy()
    best['PnL_20x'] = best['PnL_20x'].apply(lambda x: f'${x:.2f}')
    best['Duration_mins'] = best['Duration_mins'].apply(lambda x: f'{x:.0f}m')
    print(best.to_string(index=False))
    
    print(f'\n=== TOP 10 WORST TRADES — {sym} (20x) ===')
    worst = trades.nsmallest(10, 'PnL_20x')[['Trade #', 'Direction', 'Entry Time', 'Exit Time', 'Signal', 'PnL_20x', 'Duration_mins']].copy()
    worst['PnL_20x'] = worst['PnL_20x'].apply(lambda x: f'${x:.2f}')
    worst['Duration_mins'] = worst['Duration_mins'].apply(lambda x: f'{x:.0f}m')
    print(worst.to_string(index=False))
    
    verdict = 'PROFITABLE' if net_after_comm > 0 else 'UNPROFITABLE'
    print(f'\n{"="*60}')
    print(f'EXECUTIVE SUMMARY — {sym}USDT.P  (v3.7.1)')
    print(f'{"="*60}')
    print(f'  {total_trades} trades over {days} days ({total_trades/max(days,1):.1f}/day)')
    print(f'  Net P&L at 20x:      ${total_pnl:>10.2f}')
    print(f'  Commission estimate: -${commission_total:>10.2f}  (${COMM_RT} x {total_trades})')
    print(f'  NET AFTER COMMISSION: ${net_after_comm:>10.2f}')
    print(f'  Win Rate: {win_rate:.1f}% | Profit Factor: {profit_factor:.3f}')
    print(f'  Expectancy: ${expectancy:.2f}/trade')
    print(f'  Max Drawdown: ${max_dd:.2f}')
    print(f'  Max Consecutive Losses: {max_consec_losses}')
    print(f'  Worst streak damage: ${worst_streak_damage:.2f}')
    print(f'\n  VERDICT: {verdict}')

# Combined portfolio
all_combined = pd.concat(all_trades.values())
total_combined_pnl = all_combined['PnL_20x'].sum()
total_combined_trades = len(all_combined)
total_combined_comm = total_combined_trades * COMM_RT
combined_net = total_combined_pnl - total_combined_comm
combined_wr = all_combined['Win'].mean() * 100

print(f'\n{"="*60}')
print(f'COMBINED PORTFOLIO SUMMARY')
print(f'{"="*60}')
print(f'  Total trades:        {total_combined_trades}')
print(f'  Combined Win Rate:   {combined_wr:.1f}%')
print(f'  Combined P&L (20x):  ${total_combined_pnl:>10.2f}')
print(f'  Commission estimate: -${total_combined_comm:>10.2f}')
print(f'  NET AFTER COMMISSION: ${combined_net:>10.2f}')
combined_verdict = 'PROFITABLE' if combined_net > 0 else 'UNPROFITABLE'
print(f'\n  COMBINED VERDICT: {combined_verdict}')

In [ ]:
# ═══════════════════════════════════════════════════════
# MONTE CARLO — BREAKEVEN STOP SCENARIOS
# ═══════════════════════════════════════════════════════
# Commission: 0.04%/side on $10k notional = $4/side = $8 RT
# Rebate: 70% back next day → effective $2.40 RT

N_SIMS = 10_000
COMM_RT_NET = 8.0 * 0.30  # $2.40 after 70% rebate
np.random.seed(42)

# ── Fixed Scenarios (deterministic) ──
fixed_scenarios = [
    {'name': 'Perfect BE',           'trigger': 2.0, 'sl': 0.0,  'exec_rate': 1.0},
    {'name': 'Trigger $2 / Lock $1.50', 'trigger': 2.0, 'sl': 1.50, 'exec_rate': 0.90},
    {'name': 'Trigger $3 / Lock $2',    'trigger': 3.0, 'sl': 2.00, 'exec_rate': 0.85},
    {'name': 'Trigger $3 / Lock $1.50', 'trigger': 3.0, 'sl': 1.50, 'exec_rate': 0.85},
    {'name': 'Trigger $4 / Lock $2',    'trigger': 4.0, 'sl': 2.00, 'exec_rate': 0.80},
    {'name': 'No BE stop (baseline)',   'trigger': 999, 'sl': 0.0,  'exec_rate': 0.0},
]

print('=' * 90)
print('  FIXED SCENARIO ANALYSIS  (commission after 70% rebate = $2.40/RT)')
print('=' * 90)

for sym in symbols:
    trades = all_trades[sym]
    pnl = trades['PnL_20x'].values
    mfe = trades['MFE_20x'].values
    n = len(pnl)
    comm = n * COMM_RT_NET

    print(f'\n  {sym}USDT.P  ({n} trades)')
    print(f'  {"Scenario":<28} {"Adj P&L":>10} {"- Comm":>10} {"= Net":>10}  {"Saved":>6} {"EffLosers":>9}')
    print(f'  {"-"*85}')

    for sc in fixed_scenarios:
        mask = (mfe >= sc['trigger'])
        # For trades that triggered: if original P&L < stop_level, upgrade to stop_level
        adj = pnl.copy()
        triggered = mask & (np.random.random(n) < sc['exec_rate']) if sc['exec_rate'] < 1.0 else mask
        saved_mask = triggered & (pnl < sc['sl'])
        adj[saved_mask] = sc['sl']

        adj_total = adj.sum()
        net = adj_total - comm
        n_saved = saved_mask.sum()
        remaining_losers = (adj < 0).sum()

        print(f'  {sc["name"]:<28} ${adj_total:>9.0f}   -${comm:>7.0f}   ${net:>9.0f}  {n_saved:>5}   {remaining_losers:>7}')

print()

# ── Monte Carlo Simulation ──
print('=' * 90)
print(f'  MONTE CARLO SIMULATION  ({N_SIMS:,} iterations)')
print('=' * 90)
print('  Randomized per iteration:')
print('    trigger    ~ Uniform(2, 5)')
print('    stop_level ~ trigger - |Normal(0, 0.75)|  clamped [0, trigger]')
print('    exec_rate  ~ Uniform(0.70, 0.95)')
print('    slippage   ~ |Normal(0, 0.30)|  (always costs you)')
print()

mc_results = {}

for sym in symbols:
    trades = all_trades[sym]
    pnl = trades['PnL_20x'].values.astype(np.float64)
    mfe = trades['MFE_20x'].values.astype(np.float64)
    n = len(pnl)
    comm = n * COMM_RT_NET

    # Pre-allocate results
    sim_net_pnl = np.zeros(N_SIMS)
    sim_max_dd = np.zeros(N_SIMS)
    sim_saved = np.zeros(N_SIMS)

    for i in range(N_SIMS):
        # Sample scenario parameters
        trigger = np.random.uniform(2.0, 5.0)
        offset = np.abs(np.random.normal(0, 0.75))
        slippage = np.abs(np.random.normal(0, 0.30))
        stop_level = max(trigger - offset - slippage, 0.0)
        exec_rate = np.random.uniform(0.70, 0.95)

        # Per-trade execution lottery
        fires = (mfe >= trigger) & (np.random.random(n) < exec_rate)
        adj = pnl.copy()
        saved = fires & (pnl < stop_level)
        adj[saved] = stop_level

        total = adj.sum()
        sim_net_pnl[i] = total - comm
        sim_saved[i] = saved.sum()

        # Max drawdown on this sim's equity curve
        equity = np.cumsum(adj)
        running_max = np.maximum.accumulate(equity)
        dd = equity - running_max
        sim_max_dd[i] = dd.min()

    mc_results[sym] = {
        'net_pnl': sim_net_pnl,
        'max_dd': sim_max_dd,
        'saved': sim_saved,
    }

    pcts = [1, 5, 10, 25, 50, 75, 90, 95, 99]
    print(f'\n  {sym}USDT.P  ({n} trades, {N_SIMS:,} sims)')
    print(f'  {"Percentile":<12} {"Net P&L":>12} {"Max DD":>12} {"Trades Saved":>14}')
    print(f'  {"-"*52}')
    for p in pcts:
        pnl_p = np.percentile(sim_net_pnl, p)
        dd_p = np.percentile(sim_max_dd, p)
        sv_p = np.percentile(sim_saved, p)
        print(f'  {p:>3}th %ile    ${pnl_p:>10.0f}   ${dd_p:>10.0f}   {sv_p:>10.0f}')

    print(f'\n  Mean Net P&L:  ${sim_net_pnl.mean():>10.0f}')
    print(f'  Prob profitable: {(sim_net_pnl > 0).mean()*100:.1f}%')
    print(f'  Worst case (1st %ile): ${np.percentile(sim_net_pnl, 1):>10.0f}')

In [ ]:
# ═══════════════════════════════════════════════════════
# MONTE CARLO VISUALIZATIONS
# ═══════════════════════════════════════════════════════

fig_mc, axes = plt.subplots(2, 2, figsize=(20, 14), facecolor='#1a1a2e')

for col, sym in enumerate(symbols):
    res = mc_results[sym]

    # ── Top row: Net P&L Distribution ──
    ax = axes[0, col]
    ax.set_facecolor('#16213e')
    ax.hist(res['net_pnl'], bins=80, color=sym_colors[sym], alpha=0.7, edgecolor='#333')
    ax.axvline(x=0, color='white', linestyle='--', linewidth=1.0, label='Breakeven')

    p5 = np.percentile(res['net_pnl'], 5)
    p50 = np.percentile(res['net_pnl'], 50)
    p95 = np.percentile(res['net_pnl'], 95)
    ax.axvline(x=p5, color='#ff4757', linestyle='-', linewidth=1.5, label=f'5th: ${p5:,.0f}')
    ax.axvline(x=p50, color='#ffd32a', linestyle='-', linewidth=1.5, label=f'50th: ${p50:,.0f}')
    ax.axvline(x=p95, color='#00d4aa', linestyle='-', linewidth=1.5, label=f'95th: ${p95:,.0f}')

    prob_profit = (res['net_pnl'] > 0).mean() * 100
    ax.set_title(f'{sym} — Net P&L Distribution  ({prob_profit:.0f}% profitable)',
                 color='white', fontweight='bold', fontsize=12)
    ax.set_xlabel('Net P&L ($)', color='#aaa')
    ax.set_ylabel('Frequency', color='#aaa')
    ax.tick_params(colors='#aaa')
    ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=9)
    ax.grid(True, alpha=0.15)
    for spine in ax.spines.values(): spine.set_color('#333')

    # ── Bottom row: Max Drawdown Distribution ──
    ax = axes[1, col]
    ax.set_facecolor('#16213e')
    ax.hist(res['max_dd'], bins=80, color='#ff6b81', alpha=0.7, edgecolor='#333')

    dd_p50 = np.percentile(res['max_dd'], 50)
    dd_p5 = np.percentile(res['max_dd'], 5)  # worst 5%
    dd_p95 = np.percentile(res['max_dd'], 95)  # best 5%
    ax.axvline(x=dd_p50, color='#ffd32a', linestyle='-', linewidth=1.5, label=f'Median DD: ${dd_p50:,.0f}')
    ax.axvline(x=dd_p5, color='#ff4757', linestyle='-', linewidth=1.5, label=f'Worst 5%: ${dd_p5:,.0f}')

    ax.set_title(f'{sym} — Max Drawdown Distribution', color='white', fontweight='bold', fontsize=12)
    ax.set_xlabel('Max Drawdown ($)', color='#aaa')
    ax.set_ylabel('Frequency', color='#aaa')
    ax.tick_params(colors='#aaa')
    ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=9)
    ax.grid(True, alpha=0.15)
    for spine in ax.spines.values(): spine.set_color('#333')

fig_mc.suptitle(f'MONTE CARLO SIMULATION  |  {N_SIMS:,} iterations  |  BE Stop + 70% Rebate',
                color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\User\Documents\Obsidian Vault\07-TEMPLATES\monte_carlo_v371.png',
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

# ── Equity Curve Fan Chart (sampled 200 sims) ──
fig_fan, axes_fan = plt.subplots(1, 2, figsize=(20, 8), facecolor='#1a1a2e')

for col, sym in enumerate(symbols):
    ax = axes_fan[col]
    ax.set_facecolor('#16213e')

    trades = all_trades[sym]
    pnl = trades['PnL_20x'].values.astype(np.float64)
    mfe = trades['MFE_20x'].values.astype(np.float64)
    n = len(pnl)

    # Generate 200 equity curves for the fan
    n_fan = 200
    fan_curves = np.zeros((n_fan, n))
    for i in range(n_fan):
        trigger = np.random.uniform(2.0, 5.0)
        offset = np.abs(np.random.normal(0, 0.75))
        slippage = np.abs(np.random.normal(0, 0.30))
        stop_level = max(trigger - offset - slippage, 0.0)
        exec_rate = np.random.uniform(0.70, 0.95)
        fires = (mfe >= trigger) & (np.random.random(n) < exec_rate)
        adj = pnl.copy()
        adj[fires & (pnl < stop_level)] = stop_level
        fan_curves[i] = np.cumsum(adj)

    # Plot fan
    for i in range(n_fan):
        ax.plot(fan_curves[i], color=sym_colors[sym], alpha=0.04, linewidth=0.5)

    # Percentile bands
    p5 = np.percentile(fan_curves, 5, axis=0)
    p25 = np.percentile(fan_curves, 25, axis=0)
    p50 = np.percentile(fan_curves, 50, axis=0)
    p75 = np.percentile(fan_curves, 75, axis=0)
    p95 = np.percentile(fan_curves, 95, axis=0)

    x = np.arange(n)
    ax.fill_between(x, p5, p95, alpha=0.15, color=sym_colors[sym])
    ax.fill_between(x, p25, p75, alpha=0.25, color=sym_colors[sym])
    ax.plot(p50, color='#ffd32a', linewidth=2.0, label='Median')
    ax.plot(p5, color='#ff4757', linewidth=1.0, linestyle='--', label='5th %ile')
    ax.plot(p95, color='#00d4aa', linewidth=1.0, linestyle='--', label='95th %ile')

    # Original equity (no BE stop) for comparison
    ax.plot(np.cumsum(pnl), color='white', linewidth=1.0, linestyle=':', alpha=0.6, label='Original (no BE)')

    ax.axhline(y=0, color='#666', linestyle='--', linewidth=0.5)
    ax.set_title(f'{sym} — Equity Curve Fan  (200 sims)', color='white', fontweight='bold', fontsize=12)
    ax.set_xlabel('Trade #', color='#aaa')
    ax.set_ylabel('Cumulative P&L ($)', color='#aaa')
    ax.tick_params(colors='#aaa')
    ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=9)
    ax.grid(True, alpha=0.15)
    for spine in ax.spines.values(): spine.set_color('#333')

fig_fan.suptitle('EQUITY CURVE FAN CHART  |  MC Breakeven Stop Scenarios',
                 color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\User\Documents\Obsidian Vault\07-TEMPLATES\equity_fan_v371.png',
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('Monte Carlo charts saved.')